# Lesson 00 — Environment Check

## Learning Goal

Verify that your local environment is properly set up for the Graph Masterclass.

By the end of this notebook, you will:
- Know which packages are installed and their versions
- Understand Python-only mode vs Neo4j mode
- Identify any missing dependencies
- Know how to troubleshoot setup issues

**Duration**: 15 minutes

**Estimated time for full course**: 
- Foundations only (Lessons 0-8): 8 hours
- Full stack with Neo4j (Lessons 0-14): 20 hours
- Applied ML focus (Lessons 0-8, 12-13, 15-19): 16 hours
- Business analytics (Lessons 0-8, 15-19): 12 hours

## 1. Core Package Verification

Let's check that all required packages are installed and working.

In [5]:
import sys
from importlib import import_module
from packaging import version

def check_import(module_name, display_name=None):
    """Try to import a module and return status, version, and module object."""
    display_name = display_name or module_name
    try:
        mod = import_module(module_name)
        ver = getattr(mod, '__version__', 'unknown')
        return True, ver, mod
    except ImportError as e:
        return False, None, None

# Core packages (required)
core_packages = [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('networkx', 'networkx'),
    ('matplotlib', 'matplotlib'),
    ('sklearn', 'scikit-learn'),
    ('faker', 'faker'),
    ('duckdb', 'duckdb'),
]

print("\n" + "="*70)
print("CORE PACKAGES (Required)")
print("="*70)
print(f"{'Package':<20} {'Version':<20} {'Status':<20}")
print("-"*70)

core_status = {}
for module_name, display_name in core_packages:
    success, ver, mod = check_import(module_name, display_name)
    status = "✓ Installed" if success else "✗ NOT FOUND"
    ver_str = ver if success else "—"
    print(f"{display_name:<20} {ver_str:<20} {status:<20}")
    core_status[display_name] = success

print("="*70)


CORE PACKAGES (Required)
Package              Version              Status              
----------------------------------------------------------------------
pandas               2.1.4                ✓ Installed         
numpy                1.26.3               ✓ Installed         
networkx             3.2.1                ✓ Installed         
matplotlib           3.8.2                ✓ Installed         
scikit-learn         1.4.0                ✓ Installed         
faker                unknown              ✓ Installed         
duckdb               1.5.3                ✓ Installed         


## 2. Optional Package Verification

These packages enable advanced features but are not required for foundational lessons.

In [6]:
# Optional packages (for advanced lessons)
optional_packages = [
    ('pm4py', 'pm4py'),
    ('neo4j', 'neo4j'),
    ('graphdatascience', 'graphdatascience'),
]

print("\n" + "="*70)
print("OPTIONAL PACKAGES (For advanced lessons)")
print("="*70)
print(f"{'Package':<20} {'Version':<20} {'Status':<20}")
print("-"*70)

optional_status = {}
for module_name, display_name in optional_packages:
    success, ver, mod = check_import(module_name, display_name)
    if success:
        status = "✓ Installed"
    else:
        status = "⚠ Optional"
    ver_str = ver if success else "—"
    print(f"{display_name:<20} {ver_str:<20} {status:<20}")
    optional_status[display_name] = success

print("="*70)

# Check if any core packages are missing
missing_core = [name for name, status in core_status.items() if not status]
if missing_core:
    print("\n⚠️  WARNING: Missing core packages!")
    print(f"   Missing: {', '.join(missing_core)}")
    print("\n   Fix: Run 'make install' in the project root\n")
else:
    print("\n✓ All core packages installed successfully!\n")


OPTIONAL PACKAGES (For advanced lessons)
Package              Version              Status              
----------------------------------------------------------------------
pm4py                2.7.6                ✓ Installed         
neo4j                5.14.1               ✓ Installed         
graphdatascience     1.14                 ✓ Installed         

✓ All core packages installed successfully!



## 3. Python & System Information

In [7]:
import os
import platform
from pathlib import Path

print("\n" + "="*70)
print("SYSTEM INFORMATION")
print("="*70)

# Python info
print(f"Python Version    : {sys.version.split()[0]}")
print(f"Python Executable : {sys.executable}")

# OS info
print(f"Operating System  : {platform.system()} {platform.release()}")
print(f"Architecture      : {platform.machine()}")

# Paths
cwd = Path.cwd()
print(f"\nCurrent Directory : {cwd}")

# Try to get project root from Config
try:
    # Add src to path so we can import
    src_path = str(cwd / 'src') if (cwd / 'src').exists() else str(cwd.parent / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, str(cwd) if (cwd / 'src').exists() else str(cwd.parent))
    
    from src.config import Config
    print(f"Project Root      : {Config.PROJECT_ROOT}")
    print(f"Data Directory    : {Config.DATA_DIR}")
except Exception as e:
    print(f"Project Root      : Could not determine (Config import failed)")
    print(f"   Error: {str(e)}")

print("="*70)


SYSTEM INFORMATION
Python Version    : 3.12.3
Python Executable : /home/marek/Apps/graph-analysis-course/venv/bin/python
Operating System  : Linux 6.17.0-22-generic
Architecture      : x86_64

Current Directory : /home/marek/Apps/graph-analysis-course/notebooks
Project Root      : /home/marek/Apps/graph-analysis-course
Data Directory    : /home/marek/Apps/graph-analysis-course/data


## 4. Check Data Directories

In [8]:
from pathlib import Path

# Define expected directories relative to notebook location
notebook_dir = Path().resolve()
project_root = notebook_dir.parent  # Go up from notebooks/

expected_dirs = [
    project_root / 'data',
    project_root / 'data' / 'seed',
    project_root / 'data' / 'generated',
    project_root / 'data' / 'processed',
    project_root / 'notebooks',
    project_root / 'reports',
    project_root / 'src',
    project_root / 'cypher',
]

print("\n" + "="*70)
print("DIRECTORY STRUCTURE")
print("="*70)

all_exist = True
created = []

for directory in expected_dirs:
    if directory.exists():
        status = "✓ Exists"
    else:
        try:
            directory.mkdir(parents=True, exist_ok=True)
            status = "✓ Created"
            created.append(str(directory))
        except Exception as e:
            status = f"✗ Error: {str(e)}"
            all_exist = False
    
    rel_path = directory.relative_to(project_root)
    print(f"{str(rel_path):<50} {status}")

print("="*70)

if created:
    print(f"\n✓ Created {len(created)} missing directories\n")
else:
    print("\n✓ All directories present\n")


DIRECTORY STRUCTURE
data                                               ✓ Exists
data/seed                                          ✓ Exists
data/generated                                     ✓ Exists
data/processed                                     ✓ Exists
notebooks                                          ✓ Exists
reports                                            ✓ Exists
src                                                ✓ Exists
cypher                                             ✓ Exists

✓ All directories present



## 5. Environment Configuration

In [9]:
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import os

# Look for .env file
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
env_file = project_root / '.env'
env_example = project_root / '.env.example'

print("\n" + "="*70)
print("ENVIRONMENT CONFIGURATION")
print("="*70)

# Try to load .env
if env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded configuration from .env")
    env_vars = dotenv_values(env_file)
elif env_example.exists():
    print(f"ℹ Using .env.example as template (not applied)")
    env_vars = dotenv_values(env_example)
else:
    print(f"✗ No .env or .env.example found")
    env_vars = {}

# Display key settings (without revealing secrets)
print(f"\n{'Setting':<25} {'Value':<45}")
print("-"*70)

for key in ['RANDOM_SEED', 'NEO4J_ENABLED', 'NEO4J_URI', 'DEBUG', 'LOG_LEVEL']:
    value = os.getenv(key, env_vars.get(key, 'not set'))
    # Mask passwords
    if 'PASSWORD' in key or 'SECRET' in key:
        value = '****** (hidden)'
    print(f"{key:<25} {value:<45}")

print("="*70)


ENVIRONMENT CONFIGURATION
ℹ Using .env.example as template (not applied)

Setting                   Value                                        
----------------------------------------------------------------------
RANDOM_SEED               42                                           
NEO4J_ENABLED             false                                        
NEO4J_URI                 bolt://localhost:7687                        
DEBUG                     false                                        
LOG_LEVEL                 INFO                                         


## 6. Neo4j Connection Test

In [10]:
import os

print("\n" + "="*70)
print("NEO4J CONNECTION TEST")
print("="*70)

neo4j_enabled = os.getenv('NEO4J_ENABLED', 'false').lower() == 'true'
neo4j_uri = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
neo4j_user = os.getenv('NEO4J_USERNAME', 'neo4j')

print(f"\nNeo4j Mode        : {'Enabled' if neo4j_enabled else 'Disabled (Python-only mode)'}")
print(f"Neo4j URI         : {neo4j_uri}")

if not neo4j_enabled:
    print("\nℹ You are running in Python-only mode.")
    print("   This is perfect for Lessons 0-8, 12-13.")
    print("\n   To enable Neo4j mode for Lessons 9-11, 14:")
    print("   1. Edit .env and set: NEO4J_ENABLED=true")
    print("   2. Run: make neo4j-up")
    print("   3. Restart this notebook")
else:
    print("\n🔄 Attempting Neo4j connection...")
    if not optional_status.get('neo4j', False):
        print("\n✗ Neo4j Python driver not installed.")
        print("   Run: pip install neo4j")
    else:
        try:
            from neo4j import GraphDatabase
            
            neo4j_pass = os.getenv('NEO4J_PASSWORD', 'your_password_here')
            
            driver = GraphDatabase.driver(
                neo4j_uri,
                auth=(neo4j_user, neo4j_pass),
                connection_timeout=5
            )
            
            # Test connection
            with driver.session() as session:
                session.run('RETURN 1')
            
            print(f"\n✓ Connected to Neo4j at {neo4j_uri}")
            driver.close()
            
        except Exception as e:
            print(f"\n✗ Cannot connect to Neo4j at {neo4j_uri}")
            print(f"   Error: {str(e)}")
            print(f"\n   Is Neo4j running? Try: make neo4j-up")
            print(f"   Wait 10 seconds for Neo4j to start, then try again.")

print("="*70)


NEO4J CONNECTION TEST

Neo4j Mode        : Disabled (Python-only mode)
Neo4j URI         : bolt://localhost:7687

ℹ You are running in Python-only mode.
   This is perfect for Lessons 0-8, 12-13.

   To enable Neo4j mode for Lessons 9-11, 14:
   1. Edit .env and set: NEO4J_ENABLED=true
   2. Run: make neo4j-up
   3. Restart this notebook


## 7. Quick NetworkX Graph Test

Let's verify that graph creation and basic operations work.

In [11]:
import networkx as nx
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("NETWORKX GRAPH TEST")
print("="*70)

# Create a small test graph
G = nx.Graph()
G.add_edges_from([
    (1, 2), (2, 3), (3, 4), (4, 5), (5, 1),  # Pentagon
    (1, 3), (2, 4)  # Add some extra edges
])

# Compute metrics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)
avg_clustering = nx.average_clustering(G)
components = nx.number_connected_components(G)

print(f"\nGraph created successfully!")
print(f"  Nodes             : {num_nodes}")
print(f"  Edges             : {num_edges}")
print(f"  Density           : {density:.3f}")
print(f"  Avg. Clustering   : {avg_clustering:.3f}")
print(f"  Connected Comps.  : {components}")

print("\n✓ NetworkX graph operations working correctly!")
print("="*70)


NETWORKX GRAPH TEST

Graph created successfully!
  Nodes             : 5
  Edges             : 7
  Density           : 0.700
  Avg. Clustering   : 0.400
  Connected Comps.  : 1

✓ NetworkX graph operations working correctly!


## 8. Learning Modes

The Graph Masterclass offers two learning paths:

### Mode A: Python-Only ✓ (Default)

Perfect for understanding graph fundamentals with local algorithms.

**Includes**: Lessons 0-8, 12-13  
**Requires**: pandas, networkx, matplotlib, sklearn  
**Docker?**: No  
**Start**: Run `make jupyter` and open any notebook

---

### Mode B: Neo4j Full Stack

For learning persistent graph databases, Cypher queries, and enterprise-scale algorithms.

**Includes**: Lessons 0-11, 14 (optional advanced)  
**Requires**: Neo4j Community edition, Graph Data Science plugin  
**Docker?**: Yes (`docker-compose up -d neo4j`)  
**Start**:
```bash
# Edit .env:
NEO4J_ENABLED=true
# Then:
make neo4j-up
make jupyter
```

---

## Your Current Mode

In [12]:
import os

neo4j_enabled = os.getenv('NEO4J_ENABLED', 'false').lower() == 'true'

print("\n" + "="*70)
print("CURRENT LEARNING MODE")
print("="*70)

if neo4j_enabled:
    print("\nMode: NEO4J FULL STACK ✓")
    print("\nYou have access to:")
    print("  ✓ Python graph algorithms (NetworkX)")
    print("  ✓ Graph database (Neo4j)")
    print("  ✓ Cypher query language")
    print("  ✓ Graph Data Science algorithms")
    print("\nEstimated course time: 20 hours (Lessons 0-14)")
else:
    print("\nMode: PYTHON-ONLY ✓ (Recommended for first pass)")
    print("\nYou have access to:")
    print("  ✓ Graph fundamentals (Lessons 0-3)")
    print("  ✓ Core algorithms (Lessons 4-8)")
    print("  ✓ ML features (Lessons 12-13)")
    print("  ✓ Capstone projects (Lessons 15-19)")
    print("\nEstimated course time: 8-16 hours")
    print("\nTo enable Neo4j mode later:")
    print("  1. Edit .env and set: NEO4J_ENABLED=true")
    print("  2. Update NEO4J_PASSWORD if needed")
    print("  3. Run: make neo4j-up")
    print("  4. Then Lessons 9-11 become available")

print("="*70)


CURRENT LEARNING MODE

Mode: PYTHON-ONLY ✓ (Recommended for first pass)

You have access to:
  ✓ Graph fundamentals (Lessons 0-3)
  ✓ Core algorithms (Lessons 4-8)
  ✓ ML features (Lessons 12-13)
  ✓ Capstone projects (Lessons 15-19)

Estimated course time: 8-16 hours

To enable Neo4j mode later:
  1. Edit .env and set: NEO4J_ENABLED=true
  2. Update NEO4J_PASSWORD if needed
  3. Run: make neo4j-up
  4. Then Lessons 9-11 become available


## 9. Troubleshooting Guide

### Problem: ImportError for packages like pandas, numpy, etc.

**Solution**: Install dependencies
```bash
make install
```

---

### Problem: Neo4j connection refused

**Solution**: Start Neo4j container
```bash
make neo4j-up
# Wait 10 seconds for Neo4j to initialize
# Check: curl http://localhost:7474
```

---

### Problem: Port 7687 or 7474 already in use

**Solution**: Stop existing containers
```bash
docker ps  # List running containers
make neo4j-down
make neo4j-up
```

---

### Problem: DuckDB not found

**Solution**: Install optional package
```bash
pip install duckdb
```

---

### Problem: Jupyter kernel won't start

**Solution**: Reinstall kernel
```bash
python -m ipykernel install --user
jupyter lab
```

---

## Getting Help

- **Quick Start**: See [README.md](../README.md)
- **Full Curriculum**: See [MASTER_LESSONS.md](../MASTER_LESSONS.md)
- **Algorithm Reference**: See [docs/graph_algorithm_cheatsheet.md](../docs/graph_algorithm_cheatsheet.md)
- **NetworkX Reference**: See [docs/networkx_quick_reference.md](../docs/networkx_quick_reference.md)
- **Neo4j Reference**: See [docs/neo4j_quick_reference.md](../docs/neo4j_quick_reference.md)

## 10. Summary & Next Steps

---

### ✓ Environment Verification Complete!

Your local environment is ready for the Graph Masterclass. All core packages are installed and your file system is properly configured.

---

### Next: Lesson 01 — Graph Thinking and Shapes

You're ready to start! Open the next notebook:

```bash
make jupyter
# Then open: notebooks/01_graph_thinking_and_shapes.ipynb
```

**What you'll learn in Lesson 01**:
- What graphs are and why they're powerful
- Different graph shapes (chain, star, ring, complete, bipartite)
- How graph structure affects algorithms
- Real-world examples from corporate environments

**Duration**: 90 minutes

**No prerequisites**: You don't need to know graphs—we start from scratch!

---

### Course Overview

**19 lessons total** organized in 4 parts:

1. **Part I — Graph Foundations** (Lessons 0-3): What are graphs and how do they work?
2. **Part II — Core Algorithms** (Lessons 4-8): Central algorithms you need to know
3. **Part III — Neo4j & GDS** (Lessons 9-13): Enterprise-scale graph databases
4. **Part IV — Capstones** (Lessons 14-19): Real business problems solved with graphs

---

### Learning Paths

Choose based on your goals:

| Path | Time | Lessons | For | Best For |
|------|------|---------|-----|----------|
| **A** — Foundations | 8 hrs | 0-8 | Graph fundamentals | Data analysts, students |
| **B** — Full Stack | 20 hrs | 0-14 | Production graphs | Engineers, architects |
| **C** — Applied ML | 16 hrs | 0-8, 12-13, 15-19 | Graph + Machine Learning | ML engineers |
| **D** — Business | 12 hrs | 0-8, 15-19 | Graph thinking | Consultants, PMs |

See [MASTER_LESSONS.md](../MASTER_LESSONS.md) for details on each path.

---

### Success Tips

1. **Follow the sequence** — Each lesson builds on the last
2. **Don't skip exercises** — Hands-on practice is essential
3. **Run cells top-to-bottom** — Dependencies matter
4. **Experiment** — Modify code and see what happens
5. **Take notes** — Graph thinking is a new skill; consolidate your learning

---

**Ready?** Let's go to Lesson 01! 🚀